In [1]:
# ==============================================================================
# STEP 0 (PART A): INSTALL PACKAGES USING JUPYTER MAGIC (%pip)
# Why %pip? In modern Jupyter Notebook, '%pip' guarantees the package is installed
# directly into the currently running kernel, even if Windows paths have spaces!
# ==============================================================================

%pip install pymysql sqlalchemy cryptography xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ==============================================================================
# STEP 0 (PART B): AUTO-DOWNLOAD & UNZIP DATACO DATASET (BUILT-IN PYTHON)
# Why urllib & zipfile? These are 100% built into Python! We don't need any third-party
# kagglehub library. It downloads the official zip archive and extracts the CSV.
# ==============================================================================

import urllib.request
import zipfile
import os

# 1. Define target folder ('../01_raw_data') and file paths
target_folder = "../01_raw_data"
os.makedirs(target_folder, exist_ok=True)
zip_path = os.path.join(target_folder, "dataco.zip")
csv_path = os.path.join(target_folder, "DataCoSupplyChainDataset.csv")

# 2. Download the official Kaggle dataset archive if not already downloaded
if not os.path.exists(csv_path):
    print("⏳ Downloading DataCo dataset directly (approx. 30-60 seconds)...")
    url = "https://www.kaggle.com/api/v1/datasets/download/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis"
    urllib.request.urlretrieve(url, zip_path)
    print("✅ Download finished! Extracting CSV file...")
    
    # 3. Unzip 'DataCoSupplyChainDataset.csv' into '../01_raw_data'
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extract("DataCoSupplyChainDataset.csv", path=target_folder)
        
    # 4. Remove the temporary zip file to save disk space
    os.remove(zip_path)
    print(f"🎉 SUCCESS! CSV extracted to: {os.path.abspath(csv_path)}")
else:
    print(f"✅ CSV already exists at: {os.path.abspath(csv_path)}")

⏳ Downloading DataCo dataset directly (approx. 30-60 seconds)...
✅ Download finished! Extracting CSV file...
🎉 SUCCESS! CSV extracted to: D:\Supply_Chain_Analytics_Project (Oct 2026)\01_raw_data\DataCoSupplyChainDataset.csv


In [3]:
# ==============================================================================
# PHASE 1 (PART A): READ RAW CSV & SANITIZE COLUMN HEADERS FOR SQL
# Layer: Bronze Layer Preparation
# Purpose: Load the 180,519 rows into memory, handle Latin-1 character encoding
#          bugs, and standardize all 53 column names into clean SQL snake_case.
# ==============================================================================

import pandas as pd
import re
import time

# 1. Define relative path to our extracted CSV file
csv_file_path = "../01_raw_data/DataCoSupplyChainDataset.csv"

print("⏳ Step 1: Reading 180,519 rows into Pandas DataFrame...")
start_time = time.time()

# ------------------------------------------------------------------------------
# REAL-WORLD TRAP #1: CHARACTER ENCODING (INTERVIEW GOLD!)
# Why encoding='latin1'? This dataset has Spanish & Puerto Rican customer cities 
# (like Bayamón, Caguas) with special accented letters. Standard 'utf-8' crashes 
# halfway through with a UnicodeDecodeError. 'latin1' reads every row cleanly!
# ------------------------------------------------------------------------------
df_bronze = pd.read_csv(csv_file_path, encoding='latin1')

read_time = round(time.time() - start_time, 2)
print(f"✅ Loaded in {read_time} seconds! Shape: {df_bronze.shape[0]:,} rows x {df_bronze.shape[1]} columns.\n")

# ------------------------------------------------------------------------------
# STEP 2: SANITIZE COLUMN HEADERS INTO SNAKE_CASE
# Why? Raw headers have spaces and parentheses like 'Days for shipping (real)' 
# and 'Order Date (DateOrders)'. In MySQL, spaces and parentheses break queries 
# unless wrapped in backticks (`). We convert them to clean snake_case:
# Example: 'Days for shipping (real)' ---> 'days_for_shipping_real'
# ------------------------------------------------------------------------------
def clean_column_names(columns):
    cleaned = []
    for col in columns:
        col = col.strip().lower()                  # Convert to lowercase & trim outer spaces
        col = re.sub(r'[^a-z0-9]+', '_', col)      # Replace spaces, (), /, - with '_'
        col = col.strip('_')                       # Remove leading/trailing underscores
        cleaned.append(col)
    return cleaned

# Apply cleaning function to all 53 columns
df_bronze.columns = clean_column_names(df_bronze.columns)

print("📋 First 12 Cleaned Column Names (Ready for MySQL):")
print(list(df_bronze.columns[:12]))

# Preview the first 3 rows of our DataFrame
df_bronze.head(3)

⏳ Step 1: Reading 180,519 rows into Pandas DataFrame...
✅ Loaded in 1.93 seconds! Shape: 180,519 rows x 53 columns.

📋 First 12 Cleaned Column Names (Ready for MySQL):
['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email']


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_zipcode,product_card_id,product_category_id,product_description,product_image,product_name,product_price,product_status,shipping_date_dateorders,shipping_mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [5]:
# ==============================================================================
# PHASE 1 (PART B): BULK-LOAD DATAFRAME INTO MYSQL BRONZE LAYER
# Target Table: supply_chain_db.bronze_dataco_raw (180,519 rows)
# Why chunksize=10000? Pushing 180K+ rows in a single network packet can exceed 
# MySQL's 'max_allowed_packet' buffer limit and drop the connection. Sending 
# batches of 10,000 rows ensures a fast, fault-tolerant load in ~8-15 seconds.
# ==============================================================================

from sqlalchemy import create_engine
from urllib.parse import quote_plus

# 1. Configure MySQL Workbench Connection Details
DB_USER = "root"
DB_PASSWORD = quote_plus("1ox16mt400")  # <-- PUT YOUR MYSQL PASSWORD HERE
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "supply_chain_db"

# 2. Build the SQLAlchemy Connection Engine
# Note: quote_plus() safely encodes special symbols like '@' or '#' in passwords!
connection_url = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

# 3. Push the DataFrame into MySQL table 'bronze_dataco_raw'
print("⏳ Pushing 180,519 rows into MySQL table 'bronze_dataco_raw' in chunks of 10,000...")
load_start = time.time()

df_bronze.to_sql(
    name='bronze_dataco_raw',   # Table name in MySQL (Bronze Medallion Layer)
    con=engine,                 # SQLAlchemy connection object
    if_exists='replace',        # Overwrites table cleanly if we re-run the script
    index=False,                # Do not create an extra column for Pandas row index
    chunksize=10000             # Batch size per SQL INSERT statement
)

load_time = round(time.time() - load_start, 2)
print(f"🎉 BRONZE LAYER COMPLETE! All {len(df_bronze):,} rows loaded into MySQL in {load_time} seconds!")

⏳ Pushing 180,519 rows into MySQL table 'bronze_dataco_raw' in chunks of 10,000...
🎉 BRONZE LAYER COMPLETE! All 180,519 rows loaded into MySQL in 31.48 seconds!
